![Img](https://app.theheadstarter.com/static/hs-logo-opengraph.png)

# Headstarter RAG Workshop

**Skills: HuggingFace, LangChain, Pinecone**

**Other Resources:**
- [Get your Groq API Key](https://console.groq.com/keys)
- [Get your Pinecone API Key](https://www.pinecone.io/)


### What is RAG anyway?


![withoutRAG](https://github.com/user-attachments/assets/649d6101-b63a-4750-997a-b6abc25e5609)

![withRAG](https://github.com/user-attachments/assets/e6dd9c46-0bf9-4c31-bd72-a27939ef82b8)

Retrieval-Augmented Generation (RAG) is a technique primarily used in GenAI applications to improve the quality and accuracy of generated text by LLMs by combining two key processes: retrieval and generation.

### Breaking It Down:
#### Retrieval:

- Before generating a response, the system first looks up relevant information from a large database or knowledge base. This is like searching through a library or the internet to find the most useful facts, articles, or data related to the question or topic.

#### Generation:

- Once the relevant information is retrieved, the system then uses it to help generate a response. This is where the model, like GPT, creates new text (answers, explanations, etc.) based on the retrieved information.

# Install libraries

In [1]:
! pip install langchain langchain-community openai groq tiktoken pinecone-client langchain_pinecone unstructured pdfminer==20191125 pdfminer.six==20221105 pillow_heif unstructured_inference sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 30.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 46.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pdfplumber to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━

In [2]:
from langchain.document_loaders import UnstructuredPDFLoader, OnlinePDFLoader, WebBaseLoader, YoutubeLoader, DirectoryLoader, TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
from langchain_pinecone import PineconeVectorStore
from langchain.embeddings import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from google.colab import userdata
from langchain.schema import Document
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from openai import OpenAI
import numpy as np
import tiktoken
import os
from groq import Groq

pinecone_api_key = userdata.get("PINECONE_API_KEY")
os.environ['PINECONE_API_KEY'] = pinecone_api_key

# openai_api_key = userdata.get("OPENAI_API_KEY")
# os.environ['OPENAI_API_KEY'] = openai_api_key

groq_api_key = userdata.get("GROQ_API_KEY")
os.environ['GROQ_API_KEY'] = groq_api_key

/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in HuggingFaceInferenceAPIEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


# Initialize the HuggingFace Embeddings client

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

<ipython-input-3-83794808db26>:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or d

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
text = "Hello my name is Shivaji"

query_result = embeddings.embed_query(text)

In [5]:
len(query_result)

384

# Initialize the Groq client

In [6]:
# Free Llama 3.1 API via Groq

groq_client = Groq(api_key=os.getenv('GROQ_API_KEY'))

# Calculating sentence similarity with embeddings

In [7]:
def get_huggingface_embeddings(text, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    model = SentenceTransformer(model_name)
    return model.encode(text)


def cosine_similarity_between_sentences(sentence1, sentence2):
    # Get embeddings for both sentences
    embedding1 = np.array(get_huggingface_embeddings(sentence1))
    embedding2 = np.array(get_huggingface_embeddings(sentence2))

    # Reshape embeddings for cosine_similarity function
    embedding1 = embedding1.reshape(1, -1)
    embedding2 = embedding2.reshape(1, -1)

    print("Embedding for Sentence 1:", embedding1)
    print("\nEmbedding for Sentence 2:", embedding2)

    # Calculate cosine similarity
    similarity = cosine_similarity(embedding1, embedding2)
    return similarity[0][0]


# Example usage
sentence1 = "I like walking to the park"
sentence2 = "I like running to the office"


similarity = cosine_similarity_between_sentences(sentence1, sentence2)
print(f"\n\nCosine similarity between '{sentence1}' and '{sentence2}': {similarity:.4f}")

Embedding for Sentence 1: [[-7.94643245e-04 -4.52190675e-02  5.60034849e-02  4.00062427e-02
   7.82356560e-02 -3.10017494e-03  1.56902850e-01 -1.61643466e-03
   8.40177163e-02  7.29586780e-02 -2.27428079e-02 -1.00336457e-02
  -4.77766395e-02  5.78006804e-02  6.89262897e-02  2.29864451e-03
   3.41052301e-02  8.23903084e-02 -4.47455328e-03  1.18202902e-02
  -7.44136199e-02  2.10828688e-02  1.92200597e-02  5.48400506e-02
  -1.07110791e-01  8.79156739e-02 -1.64800640e-02  6.51677838e-03
  -6.67075947e-05 -4.27565910e-03 -8.20703283e-02  7.05852807e-02
  -1.80556215e-02  3.27348337e-02 -4.36549708e-02  9.93788615e-03
   5.78057729e-02 -6.92315474e-02  4.53141518e-02  4.96660061e-02
  -1.49475960e-02  5.79100400e-02  8.14057887e-02  2.62878463e-03
  -1.49136772e-02 -4.37886156e-02  2.26742886e-02 -3.19028087e-02
   1.00592643e-01  3.10834609e-02  1.30596489e-01  7.27661699e-03
   8.58718809e-03  7.95199908e-03 -7.91897438e-03  4.98278998e-03
  -8.22420716e-02  2.46388949e-02  5.11084795e-02 

# Load in the Data

Learn more about the dataset [here](https://www.kaggle.com/datasets/ayoubcherguelaine/company-documents-dataset)

In [8]:
! kaggle datasets download -d ayoubcherguelaine/company-documents-dataset
! unzip company-documents-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/ayoubcherguelaine/company-documents-dataset
License(s): apache-2.0
  0% 0.00/9.34M [00:00<?, ?B/s]
100% 9.34M/9.34M [00:00<00:00, 125MB/s]
Archive:  company-documents-dataset.zip
  inflating: CompanyDocuments/Inventory Report/monthly-Category/monthly-Category/StockReport_2016-07_1.pdf  
  inflating: CompanyDocuments/Inventory Report/monthly-Category/monthly-Category/StockReport_2016-07_2.pdf  
  inflating: CompanyDocuments/Inventory Report/monthly-Category/monthly-Category/StockReport_2016-07_3.pdf  
  inflating: CompanyDocuments/Inventory Report/monthly-Category/monthly-Category/StockReport_2016-07_4.pdf  
  inflating: CompanyDocuments/Inventory Report/monthly-Category/monthly-Category/StockReport_2016-07_5.pdf  
  inflating: CompanyDocuments/Inventory Report/monthly-Category/monthly-Category/StockReport_2016-07_6.pdf  
  inflating: CompanyDocuments/Inventory Report/monthly-Category/monthly-Category/StockReport_2016-07_7.pdf  
  inflating: 

In [9]:
def process_directory(directory_path):
    data = []
    for root, _, files in os.walk(directory_path):
        for file in files:

            file_path = os.path.join(root, file)
            print(f"Processing file: {file_path}")
            loader = PyPDFLoader(file_path)
            data.append({"File": file_path, "Data": loader.load()})

    return data

directory_path = "/content/CompanyDocuments"
documents = process_directory(directory_path)


Processing file: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10709.pdf
Processing file: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10523.pdf
Processing file: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10452.pdf
Processing file: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10264.pdf
Processing file: /content/CompanyDocuments/PurchaseOrders/purchase_orders_11075.pdf
Processing file: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10802.pdf
Processing file: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10765.pdf
Processing file: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10774.pdf
Processing file: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10337.pdf
Processing file: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10689.pdf
Processing file: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10526.pdf
Processing file: /content/CompanyDocuments/PurchaseOrders/purchase_orders_11

# Initialize Pinecone

In [10]:
# Make sure to create a Pinecone index with 384 dimensions

index_name = "rag-workshop"

namespace = "company-documents"

vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)

In [ ]:
document_data = []
for document in documents:
  document_source = document['Data'][0].metadata['source']
  docuemnt_content = document['Data'][0].page_content

  file_name = document_source.split('/')[-1]
  folder_names = document_source.split('/')[2:-1]

  doc = Document(page_content=f"<Source>\n{document_source}\n</Source>\n\n<Content>\n{document_data}\n</Content>", metadata={'source': document_source, 'file_name': file_name, "parent_folder": folder_names[-1], 'folder_names': folder_names})
  document_data.append(doc)

  print('Doc Source:', document_source)
  print('Doc Content:', docuemnt_content)


Doc Source: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10709.pdf
Doc Content: Purchase Orders
Order ID Order Date Customer Name
10709 2017-10-17 André Fonseca
Products
Product ID: Product: Quantity: Unit Price:
8 Northwoods Cranberry Sauce 40 40
51 Manjimup Dried Apples 28 53
60 Camembert Pierrot 10 34
Page 1
Doc Source: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10523.pdf
Doc Content: Purchase Orders
Order ID Order Date Customer Name
10523 2017-05-01 Hari Kumar
Products
Product ID: Product: Quantity: Unit Price:
17 Alice Mutton 25 39
20 Sir Rodney's Marmalade 15 81
37 Gravad lax 18 26
41 Jack's New England Clam Chowder 6 9.65
Page 1
Doc Source: /content/CompanyDocuments/PurchaseOrders/purchase_orders_10452.pdf
Doc Content: Purchase Orders
Order ID Order Date Customer Name
10452 2017-02-20 Jose Pavarotti
Products
Product ID: Product: Quantity: Unit Price:
28 Rössle Sauerkraut 15 36.4
44 Gula Malacca 100 15.5
Page 1
Doc Source: /content/CompanyDocuments/Purch

# Insert data into Pinecone

In [ ]:
# Make sure to install the necessary package
!pip install langchain_pinecone

# Import the PineconeVectorStore class from langchain_pinecone
from langchain_pinecone import PineconeVectorStore

vectorstore_from_texts = PineconeVectorStore.from_documents(documents=document_data, embedding=embeddings, index_name=index_name, namespace=namespace)

# Perform RAG

# Putting it all together